In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline
from tqdm.auto import tqdm
from controller import VectorStore, register_vector_control
from collections import defaultdict
import numpy as np
import torch
import pickle

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Запуск на устройстве: {DEVICE}")

Запуск на устройстве: cuda


In [ ]:
# Загружаем SD 1.5
print("Загрузка модели SD 1.5...")
pipe = StableDiffusionPipeline.from_pretrained(
    "/home/mlcore/pig_model", 
    cache_dir="/home/mlcore/cache",
    torch_dtype=torch.float32
).to(DEVICE)
pipe.safety_checker = None

In [ ]:
def compute_steering_vectors(
    pipe,
    device=DEVICE,
    concept_pos=None, 
    concept_neg=None,
    num_denoising_steps=50,
    max_prompts=None,
    n_times=1  # количество запусков одного и того же промпта
):
    """
    Вычисляет управляющие векторы для указанной модели и концепции

    Возвращает словарь векторов управления.
    Для каждого набора промптов выполняется генерация с разными seed'ами n_times раз.
    """
    print(f"Вычисляем управляющие векторы...")

    prompts_pos, prompts_neg = get_prompts_concrete(
        num=max_prompts,
        concept_pos=concept_pos,
        concept_neg=concept_neg
    )

    # Вычисляем выходы CA для промптов
    pos_vectors = []
    neg_vectors = []

    print(f"Обрабатываем {len(prompts_pos)} пар промптов, по {n_times} запусков для каждой...")

    # Проходим по каждой паре промптов
    for i, (prompt_pos, prompt_neg) in tqdm(enumerate(zip(prompts_pos, prompts_neg)), total=len(prompts_pos)):
        
        # Запускаем генерацию для одного промпта n_times раз с разными seed'ами
        for t in range(n_times):
            current_seed = i * n_times + t  # генерация уникального seed для каждого прогона
            print(f"Промпт {i+1}/{len(prompts_pos)}, запуск {t+1}/{n_times}: '{prompt_pos}' и '{prompt_neg}'")

            # Для положительного промпта
            controller = VectorStore(steer=False, device=device)
            register_vector_control(pipe.unet, controller)
            image = run_model(pipe, prompt_pos, current_seed, num_denoising_steps, device)
            pos_vectors.append(controller.vector_store)

            # Для отрицательного промпта
            controller = VectorStore(steer=False, device=device)
            register_vector_control(pipe.unet, controller)
            image = run_model(pipe, prompt_neg, current_seed, num_denoising_steps, device)
            neg_vectors.append(controller.vector_store)

    # Вычисляем управляющие векторы
    steering_vectors = {}

    # Предполагается, что количество прогонов теперь равно len(prompts_pos) * n_times
    for denoising_step in range(num_denoising_steps):
        steering_vectors[denoising_step] = defaultdict(list)

        for key in ['up', 'down', 'mid']:
            # Считаем, что структура в pos_vectors и neg_vectors одинакова,
            # и берем количество слоев по первому элементу
            num_layers = len(pos_vectors[0][denoising_step][key])
            for layer_num in range(num_layers):
                
                # Собираем векторы для текущего слоя
                pos_vectors_layer = [run_vectors[denoising_step][key][layer_num] for run_vectors in pos_vectors]
                pos_vectors_avg = np.mean(pos_vectors_layer, axis=0)

                neg_vectors_layer = [run_vectors[denoising_step][key][layer_num] for run_vectors in neg_vectors]
                neg_vectors_avg = np.mean(neg_vectors_layer, axis=0)

                # Вычисляем и нормализуем управляющий вектор
                steering_vector = pos_vectors_avg - neg_vectors_avg
                steering_vector = steering_vector / np.linalg.norm(steering_vector)

                steering_vectors[denoising_step][key].append(steering_vector)
    
    print("Управляющие векторы успешно вычислены!")
    return steering_vectors

In [ ]:
steering_vectors = compute_steering_vectors(
    pipe=pipe,
    concept_pos='horse',
    concept_neg='pig',
    num_denoising_steps=50,
    max_prompts=50,
    n_times=3,
)
with open("steering_vectors.pkl", "wb") as f:
    pickle.dump(steering_vectors, f)

with open("steering_vectors.pkl", "rb") as f:
    steering_vectors = pickle.load(f)

In [10]:
import os
import torch
import pandas as pd
from diffusers import StableDiffusionPipeline
from PIL import Image
from io import BytesIO
import base64
from steering_utils import generate_image


def image_to_base64(img: Image.Image, fmt: str = "PNG") -> str:
    buf = BytesIO(); img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode()


def prompts_to_dataframe_steered(
    pipe: StableDiffusionPipeline,
    steering_vectors,
    prompts: dict[str, str],
    seed: int,
    beta: float = 1.0,
    device: str = "cuda",
) -> pd.DataFrame:
    pipe.to(device)
    if pipe.safety_checker is not None:
        pipe.safety_checker = lambda imgs, **kw: (imgs, False)
    ids, b64 = [], []
    for k, prompt in prompts.items():
        img = generate_image(
            pipe=pipe,
            prompt=prompt,
            seed=seed,
            steering_vectors=steering_vectors,
            beta=beta,
            steer_back=True,
        )
        ids.append(k)
        b64.append(image_to_base64(img))
        print(f"{k} готов")
    return pd.DataFrame({"id": ids, "0": b64})


def save_dataframe(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False)
    print(os.path.abspath(path))

In [ ]:
import json

with open("prompts.json", "r") as file:
    prompts = json.load(file)

df = prompts_to_dataframe_steered(pipe, steering_vectors, prompts, seed=42, device=DEVICE, beta=1.0)
save_dataframe(df, "steered_images.csv")